# 입력 피처 의미·순서·전처리 실행 노트북

피처 명세와 실제 구현을 한 파일에서 확인합니다. 모델 가중치를 읽지 않고 원본 예제 입력 → 고정 보정 → 노드/거래 피처 → 2홉 그래프 텐서까지 실행합니다. 이 모델 폴더를 작업 디렉터리로 지정하고 전체 셀을 순서대로 실행하세요. 출력 폴더는 `outputs/feature_guide`입니다. GPU 전송 전 피처 계산은 CPU에서 수행합니다.

# 피처 순서와 전처리 명세

이 문서는 전달한 기존 v10/v8 가중치에 대응합니다. 진행 중인 구조 정제 재학습 가중치와 혼용하면 안 됩니다. 두 모델은 동일한 피처·보정 파일을 사용합니다.

## 노드 입력

노드 기본 피처 19개 뒤에 대상 거래 양끝 계좌 표시 ego 1개가 붙어 GPU 입력 x는 float32 `[N,20]`입니다. 인덱스는 0부터 시작합니다. 노드 집계는 context_end 날짜 전체까지 누적된 정제 거래입니다.

| 인덱스 | 이름 | 의미 | 계산 |
|---:|---|---|---|
| 0 | `log_out_count` | 송신 거래 수 log1p | ln(1+송신 거래 수) |
| 1 | `log_in_count` | 수신 거래 수 log1p | ln(1+수신 거래 수) |
| 2 | `log_out_neighbors` | 고유 수신 계좌 수 log1p | ln(1+고유 수신 계좌 수) |
| 3 | `log_in_neighbors` | 고유 송신 계좌 수 log1p | ln(1+고유 송신 계좌 수) |
| 4 | `log_self_count` | 자기 자신에게 보낸 거래 수 log1p | ln(1+자기거래 수) |
| 5 | `log_reciprocal_neighbors` | 상호 거래 계좌 수 log1p | ln(1+자기거래를 제외한 상호 송수신 상대 계좌 수) |
| 6 | `mean_paid_currency_z` | 송신 금액의 통화별 robust z 평균 | 송신 거래의 paid_currency_z 평균; 송신 없음=0 |
| 7 | `paid_outlier_fraction` | 송신 금액 중 &#124;robust z&#124; ≥ 3 비율 | 송신 거래 중 abs(paid_currency_z)>=3 비율; 송신 없음=0 |
| 8 | `log_currency_count` | 송수신 통화 종류 수 log1p | ln(1+송수신 통화 종류 수) |
| 9 | `log_active_days` | 거래가 발생한 일수 log1p | ln(1+거래 발생 날짜 수) |
| 10 | `type_Corporation` | 법인 여부 | 해당 entity_type과 정확히 일치하면 1, 아니면 0 |
| 11 | `type_Partnership` | 파트너십 여부 | 해당 entity_type과 정확히 일치하면 1, 아니면 0 |
| 12 | `type_Sole Proprietorship` | 개인사업자 여부 | 해당 entity_type과 정확히 일치하면 1, 아니면 0 |
| 13 | `type_Individual` | 개인 여부 | 해당 entity_type과 정확히 일치하면 1, 아니면 0 |
| 14 | `type_Country` | 국가 유형 여부 | 해당 entity_type과 정확히 일치하면 1, 아니면 0 |
| 15 | `type_Direct` | Direct 유형 여부 | 해당 entity_type과 정확히 일치하면 1, 아니면 0 |
| 16 | `log_entity_observed_accounts` | 같은 엔티티에서 관측된 활성 계좌 수 log1p | ln(1+동일 entity_id를 가진 스냅샷 내 활동 계좌 수); entity_id 없음=0 |
| 17 | `log_entity_out_diversity` | 엔티티의 고유 수신 엔티티 수 log1p | ln(1+송신 상대 entity_id 종류 수); 양쪽 entity_id가 있는 거래만 집계 |
| 18 | `log_entity_in_diversity` | 엔티티의 고유 송신 엔티티 수 log1p | ln(1+수신 상대 entity_id 종류 수); 양쪽 entity_id가 있는 거래만 집계 |
| 19 | `ego` | 현재 대상 거래의 송신 또는 수신 계좌 표시 | 해당 계좌면 1, 나머지 0 |

entity_type은 Corporation, Partnership, Sole Proprietorship, Individual, Country, Direct의 정확한 문자열 매칭입니다. 다른 유형과 UNKNOWN은 이 6개 열이 모두 0입니다. 단순 계좌 거래 상대 수에는 자기 계좌가 포함될 수 있고, reciprocal 계산에서는 자기거래를 제외합니다.

## 거래 입력

문맥 edge_attr와 대상 target_attr는 동일한 17열 순서이며 float32입니다. 시간차 계산은 분 단위입니다.

| 인덱스 | 이름 | 의미 | 계산 |
|---:|---|---|---|
| 0 | `pair_prior_count` | 같은 방향 계좌쌍의 1분 이상 이전 거래 수 log1p | ln(1+동일 방향 계좌쌍의 1분 이상 이전 거래 수) |
| 1 | `pair_gap_minutes` | 같은 계좌쌍 직전 거래와 간격: ln(2+분), 없으면 0 | 이전 동일 방향 계좌쌍 거래와 간격 Δ분: ln(2+Δ); 이력 없음=0 |
| 2 | `pair_count_1h` | 같은 계좌쌍의 1~60분 이전 거래 수 log1p | ln(1+동일 방향 계좌쌍의 [t−60분,t−1분] 거래 수) |
| 3 | `sender_gap_minutes` | 송신 계좌 직전 거래와 간격: ln(2+분), 없으면 0 | 동일 송신 계좌의 이전 거래와 ln(2+Δ분); 이력 없음=0 |
| 4 | `sender_count_1h` | 송신 계좌의 1~60분 이전 거래 수 log1p | ln(1+동일 송신 계좌의 [t−60분,t−1분] 거래 수) |
| 5 | `receiver_gap_minutes` | 수신 계좌 직전 거래와 간격: ln(2+분), 없으면 0 | 동일 수신 계좌의 이전 거래와 ln(2+Δ분); 이력 없음=0 |
| 6 | `receiver_count_1h` | 수신 계좌의 1~60분 이전 거래 수 log1p | ln(1+동일 수신 계좌의 [t−60분,t−1분] 거래 수) |
| 7 | `paid_currency_z` | 통화별 robust 금액 z, ±8 clipping | 통화별 robust 보정값 z; 아래 수식 참고 |
| 8 | `pair_history_count` | 해당 일 이전 동일 방향 계좌쌍 거래 수 log1p | ln(1+동일 방향 계좌쌍의 당일 0시 이전 거래 수) |
| 9 | `reverse_history_count` | 해당 일 이전 역방향 계좌쌍 거래 수 log1p | ln(1+역방향 계좌쌍의 당일 0시 이전 거래 수) |
| 10 | `pair_age_at_day_start` | 당일 0시와 동일 계좌쌍의 이전 일 마지막 거래 간격 ln(2+분) | 당일 0시와 동일 방향 계좌쌍의 이전 날짜 최종 거래 사이 Δ분: ln(2+Δ); 이력 없음=0 |
| 11 | `same_entity` | 같은 엔티티 여부: 엔티티가 없으면 0 | 송신 entity_id가 있고 수신 entity_id와 같으면 1 |
| 12 | `same_known_country` | 알려진 국가가 같으면 1 | 양쪽 country가 UNKNOWN이 아니고 동일하면 1 |
| 13 | `both_countries_known` | 양쪽 국가가 알려졌으면 1 | 양쪽 country가 UNKNOWN이 아니면 1 |
| 14 | `same_currency` | 지급 통화와 수취 통화가 같으면 1 | payment_currency == receiving_currency이면 1 |
| 15 | `is_self_transfer` | 동일 계좌로 송수신하면 1 | src == dst이면 1 |
| 16 | `amt_z_pay_ccy` | 통화별 log1p(지급금액) 표준화: (x-m)/(sd+1e-6), ±10 clipping | 지급 통화별 log금액 평균·표준편차 보정; 아래 수식 참고 |

## 금액 보정 파일과 수식

`assets/currency_stats.parquet`의 currency, scale, med, iqr를 그대로 사용합니다. CSV 사본도 제공합니다.

```text
u = ln(1 + amount / scale[currency])
z = clip((u − med[currency]) / iqr[currency], −8, 8)
```

통계는 원래 8월 1~7일 보정 구간의 통화별 지급·수취 금액에서 계산한 고정값입니다. scale은 양수 금액 중앙값, med는 변환값 u 중앙값, iqr는 max(Q75−Q25, 0.001)입니다. 보정값이 없는 통화는 z=0입니다. 원금액 0 또는 정상 거래를 의미하는 값은 아닙니다. 노드 송신 평균·이상치 비율도 이 z를 사용합니다. 환율 환산 값이 아닙니다.

`assets/amount_stats.parquet`의 payment_currency, m, sd는 별도 보정입니다.

```text
amt_z_pay_ccy = clip((ln(1 + paid) − m[currency]) / (sd[currency] + 0.000001), −10, 10)
```

m과 sd는 8월 1~7일 정제 거래의 ln(1+paid) 평균 및 모집단 표준편차입니다. m 또는 sd가 없으면 0입니다. 추론 입력으로 보정값을 다시 fitting하거나 두 종류 보정을 서로 바꾸면 안 됩니다. 전체 노드/거래 벡터에 별도의 StandardScaler는 적용하지 않습니다.

## 거래 정제·시간·그래프 처리 순서

1. 거래 ID·계좌 ID·타임스탬프·금액 유효성과 KYC 존재 여부를 검사합니다. 모델 정답 라벨은 입력하지 않습니다.
2. duplicate_group_id가 null인 거래 또는 edge_id가 대표 duplicate_group_id인 거래만 유지합니다. 이 값은 원천 적재 단계에서 지정하며, 내용이 같은 대표 거래인지 검증합니다. 임의의 반복 송금을 전부 중복으로 제거하는 기능이 아닙니다.
3. context_end 날짜까지의 거래를 문맥으로 유지합니다. SQL 이력 계산은 ORDER BY ts/day를 사용합니다. 저장 파일의 edge_id 정렬과 시간 계산 순서는 별개입니다.
4. 엣지 prior 이력은 t−1분까지, 1시간 창은 t−60분~t−1분을 포함합니다. 같은 분 거래는 prior에 포함하지 않습니다. 당일 계좌쌍 이력은 날짜 경계에서 이전 날짜만 사용합니다.
5. 노드 집계는 누적 스냅샷 전체로 계산합니다. context_end 이전 대상 거래에는 이후 거래도 문맥에 들어올 수 있으므로 실시간 인과 추론이 아닙니다.
6. 계좌별 입금·출금 각각 최대 100개를 DuckDB hash(edge_id, owner, 42, context_end 날짜), edge_id 순으로 선택합니다. timestamp 기준 최근 100개가 아닙니다.
7. 대상 송·수신 계좌에서 2홉을 확장하고 간선을 병합합니다. 이웃 제한과 2홉 때문에 패턴 전체가 남지 않을 수 있습니다.
8. 노드 ID를 로컬 인덱스로 바꾸고 ego 표시를 붙입니다. edge_index, target_index는 원본 계좌 ID가 아닌 이 로컬 인덱스입니다.
9. 방향 마스크에 따라 입금은 정방향, 출금은 역방향 메시지 전달을 수행합니다. 실제 거래 방향 자체를 바꾼 것으로 해석하지 않습니다.

시각은 Parquet timestamp(시간대 없는 값)로 공급합니다. 원천 시간대를 확인한 뒤 한 가지 규칙으로 정규화해야 하며, 시간대 정보가 없는 HI-Large를 임의로 UTC로 해석하면 안 됩니다. 계좌→정수 ID, 원 거래 ID→edge_id 매핑을 안정적으로 유지해야 고정 샘플링도 재현됩니다.

## GPU 함수 인자 순서

`model.scores(*graph.tensors(device))`의 순서는 아래와 같습니다. 모든 텐서는 동일한 device에 놓습니다.

| 위치 / NPZ 키 | 이름 | 형상 | dtype |
|---|---|---|---|
| 0 / b0 | x | N×20 | float32 |
| 1 / b1 | edge_index | 2×E | int64 |
| 2 / b2 | edge_attr | E×17 | float32 |
| 3 / b3 | target_index | 2×B | int64 |
| 4 / b4 | target_attr | B×17 | float32 |
| 5 / b5 | incoming | E | bool |
| 6 / b6 | outgoing | E | bool |

기준 예제는 B=1입니다. NPZ의 t0~t7은 원 계좌·거래 ID와 그래프 메타데이터이며 추가 입력 피처가 아닙니다. CPU에서 피처·보정 계산을 수행한 다음 결과 텐서를 GPU로 옮깁니다. 보정 Parquet를 GPU 모델에 별도 인자로 전달하지 않습니다.

## 출력

v10은 위험 점수와 검증 고정 임계값 0.4338742792606354로 알림 여부를 반환합니다. v8은 아래 9클래스 확률과 argmax를 반환하며, 보조 위험 출력의 임계값은 0.4057740867137909입니다. 클래스 순서:

```text
NORMAL, FAN-OUT, FAN-IN, G-SCATTER, S-GATHER, CYCLE, RANDOM, BIPARTITE, STACK
```

v8 NORMAL에는 무패턴 자금세탁이 포함됩니다. v8 NORMAL을 v10의 정상 판정으로 대체하지 않습니다. logits에 저장된 risk_offset/pattern_offset을 적용한 뒤 sigmoid/softmax를 사용해야 합니다. 제공된 scores()가 이를 처리합니다.


## 실행 1. 설정·명세·공통 함수

In [ ]:
from pathlib import Path
import json

# 노트북 폴더에서 실행하는 것이 기본입니다. 상위 전달 폴더에서도 실행 가능합니다.
ROOT = Path.cwd().resolve()
if not (ROOT / "models" / "nine_class.pt").is_file():
    ROOT = ROOT / "02_pattern_v8"
if not (ROOT / "models" / "nine_class.pt").is_file():
    raise FileNotFoundError("노트북이 있는 모델 폴더를 작업 디렉터리로 지정하세요.")
FAMILY = "nine_class"
import os
DEVICE = os.environ.get("AML_DEVICE", json.loads((ROOT / "settings.json").read_text()).get("device", "cpu"))


In [ ]:
"""Feature order is part of the model contract; never reorder columns."""

import json
from pathlib import Path

SCHEMA = json.loads((ROOT / "assets" / "feature_schema.json").read_text())
NODE_FEATURES = SCHEMA["node_features"]
EDGE_FEATURES = SCHEMA["edge_features"]
ENTITY_TYPES = SCHEMA["entity_types"]
API_CLASSES = SCHEMA["pattern_classes"]


In [ ]:
"""Validated relative paths and atomic JSON artifacts."""

import hashlib
import json
from pathlib import Path


In [ ]:
def resolve_path(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()


In [ ]:
def load_settings(path=None):
    settings = json.loads((Path(path) if path else ROOT / "settings.json").read_text())
    for key in ["transactions", "kyc", "output_dir"]:
        settings[key] = str(resolve_path(settings[key]))
    return settings


In [ ]:
def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


In [ ]:
q = sql_string
settings = load_settings()
settings["device"] = DEVICE
print({k: settings[k] for k in ["context_end", "target_start", "target_end", "device"]})


## 2. 고정 피처와 전처리

노드 19개·거래 17개 피처 및 보정 통계를 그대로 사용합니다. 금액 통계를 새 입력에 맞춰 재학습하지 않습니다. 정확 중복은 입력 duplicate_group_id에 지정된 대표 거래로 처리합니다. 시각 기반 이력은 날짜·시간으로 계산하고, 거래 연결은 행 위치가 아닌 edge_id를 사용합니다. 함수 정의 셀은 아래 예제 실행 전까지 데이터를 처리하지 않습니다.

In [ ]:
"""Original cumulative node feature formulas, selected into frozen 19-column order."""

import numpy as np
import pandas as pd

ALL_FEATURES = (
    [
        "log_out_count",
        "log_in_count",
        "log_out_neighbors",
        "log_in_neighbors",
        "log_self_count",
        "log_reciprocal_neighbors",
        "net_count_ratio",
        "reciprocity_ratio",
        "mean_paid_currency_z",
        "mean_received_currency_z",
        "paid_outlier_fraction",
        "received_outlier_fraction",
        "log_currency_count",
        "log_active_days",
    ]
    + ["type_" + x for x in ENTITY_TYPES]
    + [
        "log_entity_observed_accounts",
        "log_entity_out_diversity",
        "log_entity_in_diversity",
    ]
)


In [ ]:
def finite(name, x):
    if not np.isfinite(x).all():
        raise ValueError(name + " contains nonfinite values")
    return x


In [ ]:
def build_nodes(c):
    agg = c.execute(
        "WITH outgoing AS (SELECT src AS node_id,count(*) AS out_n,count(DISTINCT dst) AS out_u,\n          count(*) FILTER(WHERE src=dst) AS self_n,avg(paid_currency_z) AS out_z,\n          avg(CASE WHEN abs(paid_currency_z)>=3 THEN 1.0 ELSE 0 END) AS out_tail FROM hist GROUP BY src),\n      incoming AS (SELECT dst AS node_id,count(*) AS in_n,count(DISTINCT src) AS in_u,avg(received_currency_z) AS in_z,\n          avg(CASE WHEN abs(received_currency_z)>=3 THEN 1.0 ELSE 0 END) AS in_tail FROM hist GROUP BY dst),\n      pairs AS (SELECT DISTINCT src,dst FROM hist WHERE src!=dst),\n      reciprocal AS (SELECT a.src AS node_id,count(*) AS reciprocal_n FROM pairs a JOIN pairs b ON a.src=b.dst AND a.dst=b.src GROUP BY a.src),\n      currency_set AS (SELECT src AS node_id,payment_currency AS currency FROM hist UNION SELECT dst,receiving_currency FROM hist),\n      currencies AS (SELECT node_id,count(*) AS n_currency FROM currency_set GROUP BY node_id),\n      active AS (SELECT src AS node_id,day FROM hist UNION SELECT dst,day FROM hist),\n      active_days AS (SELECT node_id,count(*) AS n_days FROM active GROUP BY node_id),\n      ent_pairs AS (SELECT DISTINCT source_entity,target_entity FROM hist WHERE source_entity IS NOT NULL AND target_entity IS NOT NULL),\n      ent_out AS (SELECT source_entity AS entity_id,count(*) AS ent_out_n FROM ent_pairs GROUP BY source_entity),\n      ent_in AS (SELECT target_entity AS entity_id,count(*) AS ent_in_n FROM ent_pairs GROUP BY target_entity)\n      SELECT k.*,CASE WHEN k.entity_id IS NULL THEN 0 ELSE count(*) OVER(PARTITION BY k.entity_id) END AS observed_entity_accounts,coalesce(o.out_n,0) AS out_n,coalesce(i.in_n,0) AS in_n,coalesce(o.out_u,0) AS out_u,\n       coalesce(i.in_u,0) AS in_u,coalesce(o.self_n,0) AS self_n,coalesce(r.reciprocal_n,0) AS reciprocal_n,\n       coalesce(o.out_z,0) AS out_z,coalesce(i.in_z,0) AS in_z,coalesce(o.out_tail,0) AS out_tail,\n       coalesce(i.in_tail,0) AS in_tail,cu.n_currency,ad.n_days,coalesce(eo.ent_out_n,0) AS ent_out_n,\n       coalesce(ei.ent_in_n,0) AS ent_in_n\n      FROM active_days ad JOIN kyc k USING(node_id) LEFT JOIN outgoing o USING(node_id)\n      LEFT JOIN incoming i USING(node_id) LEFT JOIN reciprocal r USING(node_id)\n      JOIN currencies cu USING(node_id) LEFT JOIN ent_out eo ON k.entity_id=eo.entity_id\n      LEFT JOIN ent_in ei ON k.entity_id=ei.entity_id ORDER BY node_id"
    ).fetchdf()
    if agg.empty:
        raise ValueError("No graph history")
    v = lambda name: agg[name].to_numpy(np.float64)
    outn, inn = (v("out_n"), v("in_n"))
    ou, iu = (v("out_u"), v("in_u"))
    rec = v("reciprocal_n")
    fields = [
        np.log1p(outn),
        np.log1p(inn),
        np.log1p(ou),
        np.log1p(iu),
        np.log1p(v("self_n")),
        np.log1p(rec),
        (outn - inn) / np.maximum(outn + inn, 1),
        rec / np.maximum(ou + iu, 1),
        v("out_z"),
        v("in_z"),
        v("out_tail"),
        v("in_tail"),
        np.log1p(v("n_currency")),
        np.log1p(v("n_days")),
    ]
    fields += [(agg.entity_type == typ).to_numpy(np.float64) for typ in ENTITY_TYPES]
    fields += [
        np.log1p(v("observed_entity_accounts")),
        np.log1p(v("ent_out_n")),
        np.log1p(v("ent_in_n")),
    ]
    nx = finite("daily node features", np.stack(fields, axis=1).astype(np.float32))
    node_ids = agg.node_id.to_numpy(np.int64)
    out = pd.DataFrame(
        nx[:, [ALL_FEATURES.index(n) for n in NODE_FEATURES]], columns=NODE_FEATURES
    )
    out.insert(0, "node_id", node_ids)
    return out


In [ ]:
"""Label-free exact-deduplicated node19/edge17 preprocessing with frozen calibration.

Inputs are normalized transaction/KYC Parquet, not a vendor-specific raw CSV.
All historical transactions through context_end must be supplied. Snapshot
features include that whole day: daily close semantics, never streaming claims.
"""

from pathlib import Path
import json
import duckdb
import pandas as pd

TX_COLUMNS = [
    "edge_id",
    "ts",
    "src",
    "dst",
    "paid",
    "received",
    "payment_currency",
    "receiving_currency",
    "payment_format",
    "duplicate_group_id",
]
KYC_COLUMNS = ["node_id", "entity_id", "entity_type", "country"]


In [ ]:
def amount(column, prefix):
    return f"CASE WHEN {prefix}.scale IS NULL THEN 0 ELSE greatest(-8,least(8,(ln(1+t.{column}::DOUBLE/{prefix}.scale)-{prefix}.med)/{prefix}.iqr)) END"


In [ ]:
def validate_inputs(c, transactions, kyc):
    """Reject broken identifiers and missing KYC rather than silently losing rows."""
    c.execute(f"CREATE VIEW raw_tx AS SELECT * FROM read_parquet({q(transactions)})")
    c.execute(f"CREATE VIEW raw_kyc AS SELECT * FROM read_parquet({q(kyc)})")
    for table, columns in [("raw_tx", TX_COLUMNS), ("raw_kyc", KYC_COLUMNS)]:
        actual = set(c.execute(f"DESCRIBE {table}").fetchdf().column_name)
        if missing := (set(columns) - actual):
            raise ValueError(f"{table}: missing {sorted(missing)}")
    types = dict((row[:2] for row in c.execute("DESCRIBE raw_tx").fetchall()))
    for name in ["edge_id", "src", "dst", "duplicate_group_id"]:
        if types[name] not in ["BIGINT", "INTEGER", "SMALLINT", "TINYINT"]:
            raise ValueError(f"{name} must be a signed integer column")
    if not types["ts"].startswith("TIMESTAMP") or "WITH TIME ZONE" in types["ts"]:
        raise ValueError("ts must be a UTC-normalized timezone-naive Parquet timestamp")
    kyc_types = dict((row[:2] for row in c.execute("DESCRIBE raw_kyc").fetchall()))
    if kyc_types["node_id"] not in ["BIGINT", "INTEGER", "SMALLINT", "TINYINT"]:
        raise ValueError("KYC node_id must be a signed integer column")
    if c.execute("SELECT count(*)-count(DISTINCT edge_id) FROM raw_tx").fetchone()[0]:
        raise ValueError("edge_id must be unique")
    if c.execute("SELECT count(*)-count(DISTINCT node_id) FROM raw_kyc").fetchone()[0]:
        raise ValueError("node_id must be unique")
    bad = c.execute(
        "SELECT count(*) FROM raw_tx WHERE edge_id IS NULL OR src IS NULL OR dst IS NULL OR ts IS NULL OR edge_id<0 OR src<0 OR dst<0 OR paid IS NULL OR received IS NULL OR NOT isfinite(paid::DOUBLE) OR NOT isfinite(received::DOUBLE) OR paid<0 OR received<0 OR payment_currency IS NULL OR receiving_currency IS NULL"
    ).fetchone()[0]
    if bad:
        raise ValueError(f"{bad} transactions have invalid required fields")
    if c.execute(
        "SELECT count(*) FROM raw_kyc WHERE node_id IS NULL OR node_id<0"
    ).fetchone()[0]:
        raise ValueError("Invalid KYC node_id")
    missing = c.execute(
        "SELECT count(*) FROM (SELECT src AS node_id FROM raw_tx UNION SELECT dst FROM raw_tx) n ANTI JOIN raw_kyc k USING(node_id)"
    ).fetchone()[0]
    if missing:
        raise ValueError(f"{missing} accounts missing from KYC mapping")
    if c.execute(
        "SELECT count(*) FROM raw_tx a ANTI JOIN raw_tx b ON a.duplicate_group_id=b.edge_id WHERE a.duplicate_group_id IS NOT NULL"
    ).fetchone()[0]:
        raise ValueError("Duplicate representative missing from input history")
    fields = [
        "ts",
        "src",
        "dst",
        "paid",
        "received",
        "payment_currency",
        "receiving_currency",
        "payment_format",
    ]
    different = " OR ".join((f"a.{name} IS DISTINCT FROM b.{name}" for name in fields))
    invalid = c.execute(
        f"SELECT count(*) FROM raw_tx a JOIN raw_tx b ON a.duplicate_group_id=b.edge_id WHERE ({different}) OR (b.duplicate_group_id IS NOT NULL AND b.duplicate_group_id!=b.edge_id)"
    ).fetchone()[0]
    if invalid:
        raise ValueError(
            "Duplicate group must reference an identical retained transaction"
        )


In [ ]:
def prepare_snapshot(settings):
    """Write feature/graph Parquet and a completion manifest; no labels are read."""
    root = Path(settings["output_dir"]) / "prepared"
    if root.exists():
        raise FileExistsError(
            f"{root}: use a fresh output directory, or explicitly remove generated outputs"
        )
    end = pd.Timestamp(settings["context_end"])
    start = pd.Timestamp(settings["target_start"])
    target_end = pd.Timestamp(settings["target_end"])
    if not start <= target_end <= end:
        raise ValueError("Require target_start <= target_end <= context_end")
    if settings.get("neighbors", 100) != 100 or settings.get("sampling_seed", 42) != 42:
        raise ValueError("Frozen models require neighbors=100 and sampling_seed=42")
    root.mkdir(parents=True)
    c = duckdb.connect(str(root / "work.duckdb"))
    try:
        c.execute(
            f"SET threads={int(settings.get('threads', 2))}; SET memory_limit={q(settings.get('memory_limit', '2GB'))}"
        )
        validate_inputs(c, settings["transactions"], settings["kyc"])
        cols = ",".join(TX_COLUMNS)
        c.execute(
            f"CREATE TABLE tx AS SELECT {cols},ts::DATE AS day FROM raw_tx WHERE (duplicate_group_id IS NULL OR edge_id=duplicate_group_id) AND ts::DATE<={q(str(end.date()))} ORDER BY edge_id"
        )
        if not c.execute("SELECT count(*) FROM tx").fetchone()[0]:
            raise ValueError("Empty snapshot")
        c.execute(
            "CREATE TABLE kyc AS SELECT node_id,entity_id,coalesce(entity_type,'UNKNOWN') AS entity_type,coalesce(country,'UNKNOWN') AS country FROM raw_kyc"
        )
        c.execute(
            f"CREATE VIEW stats AS SELECT * FROM read_parquet({q(ROOT / 'assets/currency_stats.parquet')})"
        )
        c.execute(
            f"CREATE VIEW amount_stats AS SELECT * FROM read_parquet({q(ROOT / 'assets/amount_stats.parquet')})"
        )
        c.execute(
            "CREATE TABLE pair_state AS SELECT src,dst,day,\n          sum(n) OVER(PARTITION BY src,dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS history_count,\n          max(last_ts) OVER(PARTITION BY src,dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS last_ts\n          FROM (SELECT src,dst,day,count(*) n,max(ts) last_ts FROM tx GROUP BY src,dst,day)"
        )
        c.execute(
            "CREATE TABLE pairs AS SELECT t.src,t.dst,t.day,coalesce(f.history_count,0) pair_n,coalesce(r.history_count,0) reverse_n,\n          CASE WHEN f.last_ts IS NULL THEN -1 ELSE date_diff('minute',f.last_ts,t.day::TIMESTAMP) END age\n          FROM (SELECT DISTINCT src,dst,day FROM tx) t\n          ASOF LEFT JOIN pair_state f ON t.src=f.src AND t.dst=f.dst AND t.day>f.day\n          ASOF LEFT JOIN pair_state r ON t.src=r.dst AND t.dst=r.src AND t.day>r.day"
        )
        windows = []
        for name, part in [("pair", "src,dst"), ("sender", "src"), ("receiver", "dst")]:
            windows.extend(
                [
                    f"count(*) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) {name}_prior",
                    f"max(ts) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) {name}_last",
                    f"count(*) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING) {name}_hour",
                ]
            )
        c.execute(
            "CREATE TABLE temporal AS SELECT edge_id," + ",".join(windows) + " FROM tx"
        )
        gap = (
            lambda n: f"CASE WHEN z.{n}_last IS NULL THEN 0 ELSE ln(2+date_diff('minute',z.{n}_last,t.ts)) END"
        )
        expressions = [
            "ln(1+z.pair_prior)",
            gap("pair"),
            "ln(1+z.pair_hour)",
            gap("sender"),
            "ln(1+z.sender_hour)",
            gap("receiver"),
            "ln(1+z.receiver_hour)",
            amount("paid", "ps"),
            "ln(1+p.pair_n)",
            "ln(1+p.reverse_n)",
            "CASE WHEN p.age<0 THEN 0 ELSE ln(2+p.age) END",
            "CASE WHEN sk.entity_id IS NOT NULL AND sk.entity_id=dk.entity_id THEN 1 ELSE 0 END",
            "CASE WHEN sk.country!='UNKNOWN' AND dk.country!='UNKNOWN' AND sk.country=dk.country THEN 1 ELSE 0 END",
            "CASE WHEN sk.country!='UNKNOWN' AND dk.country!='UNKNOWN' THEN 1 ELSE 0 END",
            "CASE WHEN t.payment_currency=t.receiving_currency THEN 1 ELSE 0 END",
            "CASE WHEN t.src=t.dst THEN 1 ELSE 0 END",
            "CASE WHEN a.m IS NULL OR a.sd IS NULL THEN 0 ELSE greatest(-10,least(10,(ln(1+t.paid::DOUBLE)-a.m)/(a.sd+0.000001))) END",
        ]
        features = ",".join(
            (
                f'({value})::FLOAT AS "{name}"'
                for name, value in zip(EDGE_FEATURES, expressions)
            )
        )
        c.execute(
            f"CREATE TABLE edge_features AS SELECT t.edge_id,{features} FROM tx t JOIN temporal z USING(edge_id)\n          JOIN pairs p ON t.src=p.src AND t.dst=p.dst AND t.day=p.day JOIN kyc sk ON sk.node_id=t.src JOIN kyc dk ON dk.node_id=t.dst\n          LEFT JOIN stats ps ON ps.currency=t.payment_currency LEFT JOIN amount_stats a ON a.payment_currency=t.payment_currency ORDER BY t.edge_id"
        )
        c.execute(
            f"CREATE VIEW hist AS SELECT t.*,{amount('paid', 'ps')} paid_currency_z,{amount('received', 'rs')} received_currency_z,\n          ks.entity_id source_entity,kd.entity_id target_entity FROM tx t JOIN kyc ks ON ks.node_id=t.src JOIN kyc kd ON kd.node_id=t.dst\n          LEFT JOIN stats ps ON ps.currency=t.payment_currency LEFT JOIN stats rs ON rs.currency=t.receiving_currency"
        )
        nodes = build_nodes(c)
        c.register("node_features", nodes)
        adjacency = []
        for direction, owner in [("incoming", "dst"), ("outgoing", "src")]:
            adjacency.append(
                f"SELECT edge_id,{owner} AS owner,{q(direction)} AS direction FROM tx QUALIFY row_number() OVER(PARTITION BY {owner} ORDER BY hash(edge_id,{owner},42,DATE {q(str(end.date()))}),edge_id)<=100"
            )
        c.execute("CREATE TABLE adjacency AS " + " UNION ALL ".join(adjacency))
        c.execute(
            f"CREATE TABLE targets AS SELECT edge_id FROM tx WHERE day BETWEEN {q(str(start.date()))} AND {q(str(target_end.date()))} ORDER BY edge_id"
        )
        if not c.execute("SELECT count(*) FROM targets").fetchone()[0]:
            raise ValueError("No target transactions in selected dates")
        bad = " OR ".join(
            (f'"{n}" IS NULL OR NOT isfinite("{n}")' for n in EDGE_FEATURES)
        )
        if c.execute("SELECT count(*) FROM edge_features WHERE " + bad).fetchone()[0]:
            raise ValueError("Nonfinite edge features")
        orders = {
            "tx": "edge_id",
            "edge_features": "edge_id",
            "node_features": "node_id",
            "adjacency": "direction,owner,edge_id",
            "targets": "edge_id",
        }
        counts = {}
        for name, order in orders.items():
            counts[name] = c.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
            c.execute(
                f"COPY (SELECT * FROM {name} ORDER BY {order}) TO {q(root / (name + '.parquet'))} (FORMAT PARQUET, COMPRESSION ZSTD)"
            )
        manifest = dict(
            schema_version="node19_edge17_v1",
            counts=counts,
            context_end=str(end.date()),
            target_start=str(start.date()),
            target_end=str(target_end.date()),
            history_start=str(c.execute("SELECT min(day) FROM tx").fetchone()[0]),
            neighbors=100,
            sampling_seed=42,
            duckdb_version=duckdb.__version__,
            labels_used=False,
            context_policy="cumulative through context_end inclusive; not transaction-time causal",
            inputs={
                k: dict(path=settings[k], sha256=sha256(settings[k]))
                for k in ["transactions", "kyc"]
            },
            calibration={
                p.name: sha256(p)
                for p in [
                    ROOT / "assets/currency_stats.parquet",
                    ROOT / "assets/amount_stats.parquet",
                ]
            },
        )
        write_json(root / "manifest.json", manifest)
    finally:
        c.close()
    (root / "work.duckdb").unlink(missing_ok=True)
    return root


In [ ]:
def ensure_snapshot(settings):
    """Reuse only a completed snapshot with identical inputs, calibration and settings."""
    root = Path(settings["output_dir"]) / "prepared"
    if not root.exists():
        return prepare_snapshot(settings)
    manifest_path = root / "manifest.json"
    if not manifest_path.exists():
        raise ValueError("Incomplete output directory: select a fresh output_dir")
    manifest = json.loads(manifest_path.read_text())
    for key in [
        "context_end",
        "target_start",
        "target_end",
        "neighbors",
        "sampling_seed",
    ]:
        if manifest[key] != settings[key]:
            raise ValueError(
                f"Existing snapshot differs in {key}: select a fresh output_dir"
            )
    for key in ["transactions", "kyc"]:
        if manifest["inputs"][key]["sha256"] != sha256(settings[key]):
            raise ValueError(f"Changed input {key}: select a fresh output_dir")
    for name, expected in manifest["calibration"].items():
        if sha256(ROOT / "assets" / name) != expected:
            raise ValueError("Calibration changed: select a fresh output_dir")
    for name in ["tx", "edge_features", "node_features", "adjacency", "targets"]:
        if not (root / f"{name}.parquet").is_file():
            raise ValueError(f"Missing prepared artifact: {name}")
    return root


## 3. 거래별 2홉 그래프

입금·출금 방향 각각 최대 100개, sampling_seed=42를 고정합니다. 양끝 계좌를 기준으로 2홉을 확장하고 ego 표시를 붙여 노드 입력은 20차원이 됩니다. 전체 누적 그래프에 존재하는 순환도 샘플링된 입력에서는 잘릴 수 있습니다.

In [ ]:
"""On-demand, two-hop disjoint transaction graph; avoids loading full history into RAM."""

from dataclasses import dataclass
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd, torch


In [ ]:
@dataclass
class GraphInput:
    arrays: tuple
    topology: tuple
    edge_id: int
    context_end: str

    def tensors(self, device="cpu"):
        return tuple(
            (
                torch.as_tensor(
                    np.array(a, copy=True),
                    device=device,
                    dtype=(
                        torch.long
                        if i in [1, 3]
                        else torch.bool if i in [5, 6] else torch.float32
                    ),
                )
                for i, a in enumerate(self.arrays)
            )
        )

    def save(self, path):
        np.savez_compressed(
            path,
            **{f"b{i}": v for i, v in enumerate(self.arrays)},
            **{f"t{i}": v for i, v in enumerate(self.topology)},
        )


In [ ]:
def reference_graph(path, metadata):
    with np.load(path, allow_pickle=False) as data:
        return GraphInput(
            tuple((data[f"b{i}"].copy() for i in range(7))),
            tuple((data[f"t{i}"].copy() for i in range(8))),
            int(metadata["edge_id"]),
            metadata["context_end"],
        )


In [ ]:
class GraphSnapshot:

    def __init__(self, prepared):
        self.root = Path(prepared)
        self.manifest = json.loads((self.root / "manifest.json").read_text())
        if self.manifest["schema_version"] != "node19_edge17_v1":
            raise ValueError("Unsupported feature schema")
        self.c = duckdb.connect()
        self.c.execute("SET threads=2; SET memory_limit='2GB'")
        for table in ["tx", "edge_features", "node_features", "adjacency", "targets"]:
            self.c.execute(
                f"CREATE VIEW {table} AS SELECT * FROM read_parquet({q(self.root / (table + '.parquet'))})"
            )

    def close(self):
        self.c.close()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

    def target_ids(self, limit=100):
        if limit < 1:
            raise ValueError("limit must be positive")
        return (
            self.c.execute(
                f"SELECT edge_id FROM targets ORDER BY edge_id LIMIT {int(limit)}"
            )
            .fetchdf()
            .edge_id.to_numpy()
        )

    def _select(self, nodes):
        self.c.register("frontier", pd.DataFrame({"owner": np.unique(nodes)}))
        return self.c.execute(
            "SELECT a.edge_id,t.src,t.dst,bool_or(a.direction='incoming') AS incoming,bool_or(a.direction='outgoing') AS outgoing FROM adjacency a JOIN frontier f USING(owner) JOIN tx t USING(edge_id) GROUP BY a.edge_id,t.src,t.dst ORDER BY a.edge_id"
        ).fetchdf()

    def build(self, edge_id):
        target = self.c.execute(
            "SELECT t.edge_id,t.src,t.dst FROM tx t JOIN targets a USING(edge_id) WHERE t.edge_id=?",
            [int(edge_id)],
        ).fetchdf()
        if len(target) != 1:
            raise ValueError("Transaction not in declared target interval")
        src, dst = (int(target.src.iloc[0]), int(target.dst.iloc[0]))
        roots = np.unique([src, dst])
        first = self._select(roots)
        neighbors = np.unique(
            np.r_[first.src[first.incoming], first.dst[first.outgoing]]
        )
        second = self._select(neighbors)
        edges = (
            pd.concat([first, second], ignore_index=True)
            .groupby(["edge_id", "src", "dst"], as_index=False)[
                ["incoming", "outgoing"]
            ]
            .max()
            .sort_values("edge_id")
        )
        nodes = np.unique(np.r_[roots, edges.src, edges.dst])
        self.c.register("needed_nodes", pd.DataFrame({"node_id": nodes}))
        features = self.c.execute(
            "SELECT n.* FROM node_features n JOIN needed_nodes u USING(node_id) ORDER BY node_id"
        ).fetchdf()
        if not np.array_equal(features.node_id, nodes):
            raise ValueError("Missing node features")
        self.c.register(
            "needed_edges",
            pd.DataFrame({"edge_id": np.unique(np.r_[edges.edge_id, edge_id])}),
        )
        attrs = (
            self.c.execute(
                "SELECT e.* FROM edge_features e JOIN needed_edges u USING(edge_id) ORDER BY edge_id"
            )
            .fetchdf()
            .set_index("edge_id")
        )
        ego = np.isin(nodes, roots).astype(np.float32)
        x = np.column_stack([features[NODE_FEATURES].to_numpy(np.float32), ego])
        ei = np.stack(
            [np.searchsorted(nodes, edges.src), np.searchsorted(nodes, edges.dst)]
        )
        ti = np.array([[np.searchsorted(nodes, src)], [np.searchsorted(nodes, dst)]])
        arrays = (
            x,
            ei,
            attrs.loc[edges.edge_id, EDGE_FEATURES].to_numpy(np.float32),
            ti,
            attrs.loc[[edge_id], EDGE_FEATURES].to_numpy(np.float32),
            edges.incoming.to_numpy(bool),
            edges.outgoing.to_numpy(bool),
        )
        flags = arrays[5].astype(np.int64) + 2 * arrays[6].astype(np.int64)
        top = (
            nodes,
            edges.edge_id.to_numpy(np.int64),
            ei[0],
            ei[1],
            ti[0],
            ti[1],
            flags,
            ego,
        )
        return GraphInput(arrays, top, int(edge_id), self.manifest["context_end"])


## 실행 2. 입력·고정 보정값·피처 순서 확인

인덱스는 0부터 시작합니다. 노드 19열 뒤에 ego가 붙습니다. 보정값은 읽기만 하며 새 입력에 fitting하지 않습니다.

In [ ]:
from IPython.display import display
feature_dictionary = pd.DataFrame(json.loads((ROOT / "assets/feature_dictionary.json").read_text()))
display(feature_dictionary)
print("Node feature order:", list(enumerate(NODE_FEATURES)))
print("Edge feature order:", list(enumerate(EDGE_FEATURES)))
print("Ego column:", 19)
for name in ["currency_stats", "amount_stats"]:
    print(name)
    display(pd.read_parquet(ROOT / "assets" / f"{name}.parquet"))
transactions = pd.read_parquet(settings["transactions"])
kyc = pd.read_parquet(settings["kyc"])
display(transactions.head())
display(kyc.head())


## 실행 3. 전처리 실행과 생성 피처 확인

설정한 전체 문맥에서 중복을 정제하고 피처를 생성합니다. CSV는 확인용 앞 20행이며 전체 피처는 prepared 폴더의 Parquet에 저장됩니다.

In [ ]:
feature_settings = dict(settings)
feature_settings["output_dir"] = str(ROOT / "outputs/feature_guide")
prepared = ensure_snapshot(feature_settings)
node_table = pd.read_parquet(prepared / "node_features.parquet")
edge_table = pd.read_parquet(prepared / "edge_features.parquet")
assert node_table.columns.tolist() == ["node_id"] + NODE_FEATURES
assert edge_table.columns.tolist() == ["edge_id"] + EDGE_FEATURES
assert np.isfinite(node_table[NODE_FEATURES].to_numpy()).all()
assert np.isfinite(edge_table[EDGE_FEATURES].to_numpy()).all()
display(node_table.head())
display(edge_table.head())
node_table.head(20).to_csv(Path(feature_settings["output_dir"]) / "node_feature_examples.csv", index=False)
edge_table.head(20).to_csv(Path(feature_settings["output_dir"]) / "edge_feature_examples.csv", index=False)
print(json.loads((prepared / "manifest.json").read_text())["counts"])


## 실행 4. 실제 거래 1건의 전처리 수식 독립 대조

저장된 피처 한 행과 입력 거래를 직접 집계한 시간 이력·금액 보정을 비교합니다. 같은 시각 거래를 이전 이력에 포함하지 않습니다.

In [ ]:
clean = pd.read_parquet(prepared / "tx.parquet")
row = clean.iloc[0]
actual = edge_table.set_index("edge_id").loc[row.edge_id]
pair = clean[(clean.src == row.src) & (clean.dst == row.dst)]
prior = pair[pair.ts <= row.ts - pd.Timedelta(minutes=1)]
recent = prior[prior.ts >= row.ts - pd.Timedelta(minutes=60)]
expected_prior = np.log1p(len(prior))
expected_hour = np.log1p(len(recent))
expected_gap = np.log(2 + int((row.ts - prior.ts.max()).total_seconds() // 60)) if len(prior) else 0.0
stats = pd.read_parquet(ROOT / "assets/currency_stats.parquet").set_index("currency")
if row.payment_currency in stats.index:
    stat = stats.loc[row.payment_currency]
    expected_amount = np.clip((np.log1p(float(row.paid) / stat.scale) - stat.med) / stat.iqr, -8, 8)
else:
    expected_amount = 0.0
expected = [expected_prior, expected_hour, expected_gap, expected_amount]
observed = actual[["pair_prior_count", "pair_count_1h", "pair_gap_minutes", "paid_currency_z"]].to_numpy(dtype=float)
np.testing.assert_allclose(observed, expected, rtol=1e-6, atol=1e-6)
print("Independent feature formula check passed:", int(row.edge_id))


## 실행 5. GPU 모델에 넘길 텐서 순서·자료형

여기서는 CPU 텐서로 shape와 dtype을 검사합니다. GPU에서는 동일한 graph에 `graph.tensors("cuda")`를 호출하면 됩니다. 모델 실행은 `00_weights_only.ipynb` 또는 전체 추론 노트북을 사용하세요.

In [ ]:
with GraphSnapshot(prepared) as snapshot:
    target_id = int(snapshot.target_ids(1)[0])
    graph = snapshot.build(target_id)
tensors = graph.tensors("cpu")
names = ["x", "edge_index", "edge_attr", "target_index", "target_attr", "incoming", "outgoing"]
dtypes = [torch.float32, torch.int64, torch.float32, torch.int64, torch.float32, torch.bool, torch.bool]
assert tensors[0].shape[1] == 20
assert tensors[2].shape[1] == tensors[4].shape[1] == 17
assert tensors[1].shape[0] == tensors[3].shape[0] == 2
assert all(t.dtype == dt for t, dt in zip(tensors, dtypes))
rows = [{"position": i, "name": name, "shape": list(t.shape), "dtype": str(t.dtype)} for i, (name, t) in enumerate(zip(names, tensors))]
display(pd.DataFrame(rows))
graph.save(Path(feature_settings["output_dir"]) / "example_graph.npz")
write_json(Path(feature_settings["output_dir"]) / "feature_validation.json", {"passed": True, "node_features": 19, "ego_column": 19, "edge_features": 17, "manual_formula_checks": 4, "target_edge_id": target_id, "tensor_contract": rows})
